# Prediction-Informed Experiments

This notebook implements a **sequential D-optimal experimental design** workflow for binary alloy liquidus measurements.

## Workflow
1. Generate an initial liquidus prediction (ML model or DFT pseudo-constraint fit).
2. Use the Fisher Information Matrix (FIM) to identify the composition where a single new measurement would maximize information about the model parameters.
3. Record your measured liquidus temperature at the recommended composition.
4. Refit the model with the accumulated measurements.
5. Recompute the FIM and recommend the next composition.
6. Repeat steps 3–5 as many times as desired.

## How to use
- Run cells **0–6** once to initialize.
- For each measurement cycle: fill in **cell 8** (your measured x and T), then run **cell 9**.
- Run **cell 10** at any time for a summary of information gain across all iterations.

### Key concept: D-optimal score
For a candidate composition x, the D-optimal score is:
$$\text{score}(x) = 1 + \mathbf{j}(x)^\top \mathbf{I}^{-1} \mathbf{j}(x) / \sigma^2$$
where $\mathbf{j}(x) = \partial T_{\text{liq}}(x) / \partial \boldsymbol{\theta}$ is the Jacobian of the predicted liquidus temperature w.r.t. the free model parameters. A higher score means measuring at that composition provides more information. For the first measurement (no prior data), this reduces to $1 + \|\mathbf{j}(x)\|^2 / \lambda_{\text{prior}}$, ranking by sensitivity.

In [1]:
import os
import copy
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
import gliquid.config as cfg

from gliquid.binary import BinaryLiquid, BLPlotter
from gliquid.fisher_information import compute_fim, find_optimal_next_measurement

# Required for fetching any uncached MP data.
os.environ["NEW_MP_API_KEY"] = "Jcw46im7UV1xOfHzbZZ8nkq8BH00Pf6s"

In [2]:
# ============================================================
# CONFIGURATION — edit these values for your system
# ============================================================

SYSTEM = "Cu-Mg"           # Target binary system (e.g. 'Ga-Ru', 'Cu-Mg')
USE_ML_PREDICTION = True   # True = ML ProductionModelRunner; False = DFT pseudo-constraint fit

SIGMA        = 5.0         # Assumed temperature measurement uncertainty (K)
PRIOR_LAMBDA = 1e-4        # Regularization for cold-start (first iteration, no data yet)
PARAM_FORMAT = 'comb-exp'  # Parameter format for BinaryLiquid

# Candidate compositions to evaluate for the next measurement (atomic fraction of component B)
CANDIDATE_GRID = np.linspace(0.05, 0.95, 50)

print(f"System: {SYSTEM}  |  Prediction: {'ML model' if USE_ML_PREDICTION else 'DFT pseudo-constraint'}")
print(f"σ = {SIGMA} K  |  prior_λ = {PRIOR_LAMBDA}")

System: Cu-Mg  |  Prediction: ML model
σ = 5.0 K  |  prior_λ = 0.0001


In [3]:
# ============================================================
# STEP 1: Load system and generate initial prediction
# ============================================================

bl = BinaryLiquid.from_cache(SYSTEM, param_format=PARAM_FORMAT)
bl.comp_range_fit_lim = 0.0   # allow sparse composition ranges
bl.init_error = False

if USE_ML_PREDICTION:
    from gliquid.production_model_runner import ProductionModelRunner
    runner = ProductionModelRunner(str(cfg.data_dir) + "/20260217_135723")
    # predict_system returns [L0_a, L0_b, L1_a]; L1_b = 0 for comb-exp format
    pred_params = runner.predict_system(SYSTEM) + [0]
    bl.update_params(pred_params)
    source = "ML model"
else:
    bl.fit_parameters(disable_inv_constrs=True, n_opts=1, max_iter=64, verbose=False)
    pred_params = bl.get_params()
    source = "DFT pseudo-constraint"

# Store initial params for warm-starting refits
initial_params = list(pred_params)

print(f"Initial prediction ({source}) for {SYSTEM}:")
print(f"  L0_a = {pred_params[0]:.1f}  L0_b = {pred_params[1]:.4f}")
print(f"  L1_a = {pred_params[2]:.1f}  L1_b = {pred_params[3]:.4f}")

Cu: H_liq = 13260 J/mol, S_liq = 9.7660 J/(mol·K), T_fusion = 1357.77 K, polymorphs = 0
Mg: H_liq = 8480 J/mol, S_liq = 9.1874 J/(mol·K), T_fusion = 923 K, polymorphs = 0

Reading MPDS json from entry at https://mpds.io/entry/C906729...



c:\Users\willwerj\miniforge3\envs\gliquidenv\lib\pickle.py:1718: UserWarning: [15:24:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


Initial prediction (ML model) for Cu-Mg:
  L0_a = -40762.2  L0_b = -4.2161
  L1_a = -9370.7  L1_b = 0.0000


In [4]:
# ============================================================
# STEP 2: Plot initial predicted phase diagram
# ============================================================

BLPlotter(bl).show('pred')

In [5]:
# ============================================================
# STEP 3: Initialize measurement tracking
# ============================================================

# Each entry: [atomic_fraction_B, temperature_K]
measured_liquidus = []

# History for summary plot: one entry per measurement added
iteration_history = []  # list of dicts: {'n', 'det_fim', 'param_variances', 'param_names', 'params'}

print("Measurement tracking initialized. 'measured_liquidus' is empty.")
print("Run cell 6 to get your first measurement recommendation.")

Measurement tracking initialized. 'measured_liquidus' is empty.
Run cell 6 to get your first measurement recommendation.


In [6]:
# ============================================================
# STEP 4: Cold-start FIM — recommend FIRST measurement
#         (Run this once before any measurements are taken)
# ============================================================

# With no measurements, the FIM is zero. prior_lambda adds a small identity
# matrix so the score = 1 + ||j(x)||² / prior_lambda, ranking by sensitivity.
fim_cold = compute_fim(
    bl,
    x_compositions=np.array([]),
    sigma=SIGMA,
    prior_lambda=PRIOR_LAMBDA,
)

opt_cold = find_optimal_next_measurement(fim_cold, bl, candidate_x=CANDIDATE_GRID)

print(f"=== Iteration 0: No measurements yet ===")
print(f"Free parameters: {fim_cold.param_names}")
print()
print("Top 5 recommended first measurement compositions:")
print(f"  {'Rank':<6} {'x (at. frac.)':<18} {'D-optimal score':<18}")
print(f"  {'-'*44}")
for rank, (x, score) in enumerate(zip(opt_cold.ranked_x[:5], opt_cold.d_optimal_scores[:5]), 1):
    marker = "  <-- RECOMMENDED" if rank == 1 else ""
    print(f"  {rank:<6} {x:<18.4f} {score:<18.4f}{marker}")

# Visualize D-optimal scores across composition range
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=opt_cold.ranked_x, y=opt_cold.d_optimal_scores,
    mode='markers', marker=dict(color='steelblue', size=8),
    name='D-optimal score'
))
fig.add_vline(
    x=opt_cold.ranked_x[0], line_dash='dash', line_color='red',
    annotation_text=f"Recommended: x = {opt_cold.ranked_x[0]:.3f}",
    annotation_position='top right'
)
fig.update_layout(
    title=f"{SYSTEM}: D-optimal scores for first measurement",
    xaxis_title=f"Composition (at. frac. {SYSTEM.split('-')[1]})",
    yaxis_title="D-optimal score (∝ ||∂T/∂θ||²)",
    template='simple_white',
)
fig.show()

=== Iteration 0: No measurements yet ===
Free parameters: ['L0_a', 'L0_b', 'L1_a', 'L1_b']

Top 5 recommended first measurement compositions:
  Rank   x (at. frac.)      D-optimal score   
  --------------------------------------------
  1      0.2153             94334.1347          <-- RECOMMENDED
  2      0.2337             94081.8237        
  3      0.2520             93790.9467        
  4      0.2704             93510.8100        
  5      0.4173             93503.4336        


---
## Measurement Loop

**For each new measurement, run cells 8 and 9 in sequence:**

1. **Cell 8**: Enter the composition and temperature of your new measurement.
2. **Cell 9**: Run this cell — it refits the model, updates the FIM, and recommends your next measurement.

Repeat as many times as desired. Run cell 10 at any point for a summary of how the FIM evolves.

In [7]:
# ============================================================
# [LOOP] Enter your measurement result here
# ============================================================

# Edit these two values, then run this cell followed by cell 9.
new_x_atfrac  = 0.05     # Composition of component B (atomic fraction, 0–1)
new_T_celsius = 1008.8   # Measured liquidus temperature (°C)

# ============================================================
new_point = [new_x_atfrac, new_T_celsius + 273.15]
measured_liquidus.append(new_point)
print(f"Recorded measurement #{len(measured_liquidus)}:")
print(f"  x({SYSTEM.split('-')[1]}) = {new_x_atfrac:.4f}  →  T = {new_T_celsius:.1f} °C  ({new_T_celsius + 273.15:.2f} K)")
print(f"Total measurements so far: {len(measured_liquidus)}")

Recorded measurement #1:
  x(Mg) = 0.0500  →  T = 1008.8 °C  (1281.95 K)
Total measurements so far: 1


In [8]:
# ============================================================
# [LOOP] Refit model and update FIM
# ============================================================

n = len(measured_liquidus)
assert n > 0, "No measurements recorded. Run cell 8 first."

# --- Refit with accumulated measurements ---
# Use a deepcopy so the base bl (initial prediction params) is preserved
bl_fit = copy.deepcopy(bl)
bl_fit.digitized_liq  = list(measured_liquidus)
bl_fit.max_liq_temp   = max(pt[1] for pt in measured_liquidus)
bl_fit.min_liq_temp   = min(pt[1] for pt in measured_liquidus)
bl_fit.comp_range_fit_lim = 0.0
bl_fit.init_error     = False

bl_fit.fit_parameters(
    disable_inv_constrs=True,
    allow_sparse_data=True,
    check_phase_mismatch=False,
    n_opts=1,
    max_iter=128,
    verbose=False,
    params_init=initial_params,
)
current_params = bl_fit.get_params()

# --- Recompute FIM with new data ---
x_measured = np.array([pt[0] for pt in measured_liquidus])
fim = compute_fim(
    bl_fit,
    x_compositions=x_measured,
    sigma=SIGMA,
    prior_lambda=PRIOR_LAMBDA,
)

# --- Find next recommended measurement ---
opt = find_optimal_next_measurement(fim, bl_fit, candidate_x=CANDIDATE_GRID)

# --- Store in history ---
iteration_history.append({
    'n': n,
    'det_fim': fim.det_fim,
    'param_variances': fim.param_variances.copy(),
    'param_names': fim.param_names,
    'params': current_params,
})

# --- Print summary ---
print(f"=== Iteration {n}: {n} measurement(s) ===")
print(f"  Fitted parameters:")
for name, val in zip(['L0_a', 'L0_b', 'L1_a', 'L1_b'], current_params):
    print(f"    {name} = {val:.4f}")
print(f"  Parameter 1σ uncertainties (data-driven):")
for name, var in zip(fim.param_names, fim.param_variances):
    print(f"    {name}: ±{np.sqrt(var):.4f}")
print(f"  det(FIM) = {fim.det_fim:.4e}")
print()
print("Top 5 recommended NEXT measurement compositions:")
print(f"  {'Rank':<6} {'x (at. frac.)':<18} {'D-optimal score':<18}")
print(f"  {'-'*44}")
for rank, (x, score) in enumerate(zip(opt.ranked_x[:5], opt.d_optimal_scores[:5]), 1):
    marker = "  <-- RECOMMENDED" if rank == 1 else ""
    print(f"  {rank:<6} {x:<18.4f} {score:<18.4f}{marker}")

# --- Plot updated phase diagram ---
blp_current = BLPlotter(bl_fit)
fig_pd = blp_current.get_plot('pred')

# Overlay measured points
fig_pd.add_trace(go.Scatter(
    x=[pt[0] * 100 for pt in measured_liquidus],
    y=[pt[1] - 273.15 for pt in measured_liquidus],
    mode='markers',
    marker=dict(color='cornflowerblue', size=12, symbol='square',
                line=dict(width=1, color='black')),
    name='Measured liquidus'
))
fig_pd.add_vline(
    x=opt.ranked_x[0] * 100, line_dash='dash', line_color='red',
    annotation_text=f"Next: x = {opt.ranked_x[0]:.3f}"
)
fig_pd.update_layout(title=f"{SYSTEM}: Prediction after {n} measurement(s)")
fig_pd.show()

# --- D-optimal score plot ---
fig_d = go.Figure()
fig_d.add_trace(go.Scatter(
    x=opt.ranked_x, y=opt.d_optimal_scores,
    mode='markers', marker=dict(color='steelblue', size=8), name='D-optimal score'
))
for pt in measured_liquidus:
    fig_d.add_vline(x=pt[0], line_dash='dot', line_color='gray', line_width=1)
fig_d.add_vline(
    x=opt.ranked_x[0], line_dash='dash', line_color='red',
    annotation_text=f"Recommended: x = {opt.ranked_x[0]:.3f}",
    annotation_position='top right'
)
fig_d.update_layout(
    title=f"{SYSTEM}: D-optimal scores after {n} measurement(s)",
    xaxis_title=f"Composition (at. frac. {SYSTEM.split('-')[1]})",
    yaxis_title="D-optimal score",
    template='simple_white',
)
fig_d.show()


Maximum composition range fitted: [0.05, 0.05]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
=== Iteration 1: 1 measurement(s) ===
  Fitted parameters:
    L0_a = -35072.8498
    L0_b = -3.5145
    L1_a = -5593.6215
    L1_b = 0.0000
  Parameter 1σ uncertainties (data-driven):
    L0_b: ±15.3914
    L1_a: ±99.9998
  det(FIM) = 4.2221e-07

Top 5 recommended NEXT measurement compositions:
  Rank   x (at. frac.)      D-optimal score   
  --------------------------------------------
  1      0.5459             2353.6272           <-- RECOMMENDED
  2      0.5276             2352.6307         
  3      0.5643             2350.5042         
  4      0.5092             2348.4435         
  5      0.5827             2342.3286         


In [9]:
# ============================================================
# SUMMARY: FIM evolution across all iterations
# (Run at any time after completing one or more iterations)
# ============================================================

assert len(iteration_history) > 0, "No iterations completed yet. Run cells 8 and 9 at least once."

n_iters     = [h['n'] for h in iteration_history]
param_names = iteration_history[0]['param_names']
n_params    = len(param_names)
colors      = ['#4477AA', '#EE6677', '#228833', '#CCBB44']

fig = sp.make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Parameter 1σ uncertainty vs. number of measurements",
        "det(FIM) vs. number of measurements",
    ],
)

for i, name in enumerate(param_names):
    stds = [np.sqrt(h['param_variances'][i]) for h in iteration_history]
    fig.add_trace(
        go.Scatter(x=n_iters, y=stds, name=name, mode='lines+markers',
                   line=dict(color=colors[i % len(colors)])),
        row=1, col=1
    )

det_fims = [h['det_fim'] for h in iteration_history]
fig.add_trace(
    go.Scatter(x=n_iters, y=det_fims, name='det(FIM)', mode='lines+markers',
               line=dict(color='black')),
    row=1, col=2
)

fig.update_xaxes(title_text="Number of measurements", dtick=1)
fig.update_yaxes(title_text="1σ uncertainty", row=1, col=1)
fig.update_yaxes(title_text="det(FIM)", row=1, col=2)
fig.update_layout(
    title=f"{SYSTEM}: Information gain over sequential measurements",
    template='simple_white',
    height=450,
)
fig.show()

# Print parameter values across iterations
print(f"{'Iter':<6} {'L0_a':<12} {'L0_b':<10} {'L1_a':<12} {'L1_b':<10} ", end='')
for name in param_names:
    print(f"{'σ('+name+')':<12}", end='')
print()
print('-' * (50 + 12 * n_params))
for h in iteration_history:
    p = h['params']
    print(f"{h['n']:<6} {p[0]:<12.1f} {p[1]:<10.4f} {p[2]:<12.1f} {p[3]:<10.4f} ", end='')
    for var in h['param_variances']:
        print(f"{np.sqrt(var):<12.4f}", end='')
    print()

Iter   L0_a         L0_b       L1_a         L1_b       σ(L0_b)     σ(L1_a)     
--------------------------------------------------------------------------
1      -35072.8     -3.5145    -5593.6      0.0000     15.3914     99.9998     
